# Remove Control File Duplicates

**Purpose:** Remove duplicates from control file

In [ ]:
data_path = "/Users/Mach/dev/aps/data/2026_Dmodel_data"
input_file = "aux_dataset_car_vindate_ctl.parquet"
output_file = "aux_dataset_car_vindate_ctl_no_duplicate.parquet"
key_column = "vin_date"

In [ ]:
import pandas as pd
import os
print("="*60)
print("REMOVE DUPLICATES")
print("="*60)

In [ ]:
input_path = f"{data_path}/{input_file}"
print(f"Loading: {input_path}")
df = pd.read_parquet(input_path)
print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
print(f"Unique {key_column}: {df[key_column].nunique():,}")

In [ ]:
total = len(df)
unique = df[key_column].nunique()
print(f"\nDuplicates to remove: {total - unique:,}")
print(f"Ratio: {total/unique:.2f}x")

In [ ]:
print("\n* Verifying duplicates are identical...")
dup_mask = df.duplicated(subset=key_column, keep=False)
dup_df = df[dup_mask]
all_ok = True
for col in df.columns:
    if col == key_column: continue
    nunique = dup_df.groupby(key_column)[col].nunique()
    bad = (nunique > 1).sum()
    if bad > 0:
        print(f"  {col}: {bad} keys differ")
        all_ok = False
if all_ok: print("✓ All duplicates identical")

In [ ]:
print("\n* Removing duplicates...")
df_clean = df.drop_duplicates(subset=key_column, keep="first")
print(f"  Before: {len(df):,}")
print(f"  After: {len(df_clean):,}")
print(f"  Removed: {len(df) - len(df_clean):,}")

In [ ]:
print("\n* Verification:")
remaining = df_clean[key_column].duplicated().sum()
print(f"  Remaining duplicates: {remaining}")
if remaining == 0: print("  ✓ No duplicates")

In [ ]:
output_path = f"{data_path}/{output_file}"
print(f"\n* Saving: {output_path}")
df_clean.to_parquet(output_path, index=False)
size_mb = os.path.getsize(output_path) / (1024**2)
print(f"  Size: {size_mb:.1f} MB")
print("  ✓ Saved")

In [ ]:
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"\nInput:  {input_file}")
print(f"  {len(df):,} rows")
print(f"\nOutput: {output_file}")
print(f"  {len(df_clean):,} rows")
print(f"\nReduction: {(1-len(df_clean)/len(df))*100:.1f}%")
print(f"\n✓ Update config to: {output_file}")